# 03 — Signal sensitivity per plane

For each candidate plane (survivors of gates 1–2, even half): signal efficiency into
A, xsec-independent sideband leakage (S_B/S_A etc.), the bias the leakage induces in
the background prediction as a function of signal strength, and the Asimov Z at the
1 fb reference with the notebook-02 non-closure systematic folded in. Signal
normalization uses the census gen-filter denominators (absolute ε is honest).

In [1]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

sys.path.insert(0, os.getcwd())
import abcd_tools as at
import study_setup as ss

hep.style.use("CMS")
plt.rcParams["figure.figsize"] = (9, 7)
# even-parity half of the sample = half the effective luminosity
LUMI_LABEL = r"29.9 fb$^{-1}$ (13 TeV, 2018 sim., even half)"

def cms_label(ax=None):
    hep.cms.label("Work in progress", data=False, rlabel=LUMI_LABEL, ax=ax)

# pre-skim / pre-filter sums of gen weights (see README "Normalization")
SUMW_PRE = {}
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_2MU2E))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_4MU))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_BKG_UNSKIMMED))
SUMW_PRE.update(ss.SUMW_PRE_OVERRIDES)   # DY rogue-file repair (see study_setup)

# normalization tripwire: a factor outside [1e-5, 1] means a broken denominator
for _s in ss.ANALYSIS_BACKGROUNDS:
    _f = ss.fetch(_s)["metadata"]["scaled_sum_weights"] / SUMW_PRE[_s] / ss.FW.get(_s, 1.0)
    assert 1e-5 < _f <= 1.0, f"normalization factor out of range for {_s}: {_f:.3g}"
SUMW_PRE.update(ss.SUMW_PRE_OVERRIDES)   # DY rogue-file repair (see study_setup)

# normalization tripwire: a factor outside [1e-5, 1] means a broken denominator
for _s in ss.ANALYSIS_BACKGROUNDS:
    _f = ss.fetch(_s)["metadata"]["scaled_sum_weights"] / SUMW_PRE[_s] / ss.FW.get(_s, 1.0)
    assert 1e-5 < _f <= 1.0, f"normalization factor out of range for {_s}: {_f:.3g}"

# Background sums built ONE SAMPLE AT A TIME (holding all 44 samples in memory OOMs
# the interactive node); signals are loaded lazily where needed, one at a time.
# TTJets kept at the campaign 471.7 pb; the NNLO alternative is an explicit rescale.
total_bkg, by_process = ss.accumulate_normalized(list(ss.ANALYSIS_BACKGROUNDS), SUMW_PRE)
def load_sig(s):
    return ss.load_normalized(s, SUMW_PRE)[0]
print(f"accumulated {len(ss.BACKGROUNDS)} backgrounds; scan hists: {len(total_bkg)}")

accumulated 18 backgrounds; scan hists: 8


In [2]:
# Sensitivity per plane x prescription x signal point. The background prediction at
# the SR working point uses B*C/D from the (sparse) tight regions; its uncertainty
# folds the MC-stat error with the non-closure systematic from the notebook-02
# projection: syst = |R_proj - 1| (+) R_proj_err. Planes without a valid projection
# are skipped (nothing defensible to quote).
gates = json.load(open(os.path.join(ss.WORKDIR, "gates_even.json")))
results = {}
for ch, signals in [("2mu2e", ss.SIGNALS_2MU2E), ("4mu", ss.SIGNALS_4MU)]:
    for pname, spec in ss.PLANES[ch].items():
        for presc in ["i", "iii"]:
            g3 = gates["gate3"].get(f"{ch}/{pname}/{presc}") or {}
            if "syst" not in g3:
                continue
            syst = g3["syst"]
            lo = dict(xlo=0.0, ylo=0.0) if presc == "iii" else {}
            bvals, bvar, xe, ye = ss.plane_arrays(total_bkg, ch, pname, parity=0)
            breg = at.region_sums(bvals, bvar, xe, ye, spec["xspec"], spec["yspec"], **lo)
            bpred, bpred_var = at.abcd_prediction(breg)
            if not np.isfinite(bpred):
                continue
            sigma_b = np.sqrt(max(bpred_var, 0) + (syst * bpred) ** 2)
            for s in signals:
                svals, svar, sxe, sye = ss.plane_arrays(load_sig(s), ch, pname, parity=0)
                sreg = at.region_sums(svals, svar, sxe, sye, spec["xspec"], spec["yspec"], **lo)
                results[f"{ch}/{pname}/{presc}/{s}"] = {
                    "S_A": sreg["A"][0], "B_pred": bpred, "sigma_B": sigma_b,
                    "Z": at.asimov_z(sreg["A"][0], max(bpred, 1e-4), sigma_b),
                    "leakage": at.leakage_ratios(sreg)}
json.dump(results, open(os.path.join(ss.WORKDIR, "sensitivity_even.json"), "w"),
          indent=1, default=float)
print(f"{len(results)} plane x prescription x signal evaluations")
print("NOTE: Z values use tight-region B*C/D predictions from the n_eff~1 regime -")
print("indicative ranking only, NOT calibrated significances (see notebook 02).")

# prediction bias vs signal strength for the leading plane (its C sideband hosts the
# proposed low-mass SR, so signal leakage there matters most)
spec = ss.PLANES["2mu2e"]["P4_muiso_mjj"]
bvals, bvar, xe, ye = ss.plane_arrays(total_bkg, "2mu2e", "P4_muiso_mjj", parity=0)
breg = at.region_sums(bvals, bvar, xe, ye, spec["xspec"], spec["yspec"])
print("P4 leakage per signal point (S_region/S_A) and prediction bias vs mu:")
for s in ss.SIGNALS_2MU2E:
    sv, sw, sxe, sye = ss.plane_arrays(load_sig(s), "2mu2e", "P4_muiso_mjj", parity=0)
    sreg = at.region_sums(sv, sw, sxe, sye, spec["xspec"], spec["yspec"])
    lr = at.leakage_ratios(sreg)
    bias = at.prediction_bias_vs_mu(breg, sreg, [1.0])[0][1]
    fmt = {k: (f"{v:.3f}" if np.isfinite(v) else "-") for k, v in lr.items()}
    print(f"  {s:34s} B/A={fmt['B']} C/A={fmt['C']} D/A={fmt['D']} "
          f"bias(mu=1)={bias if np.isfinite(bias) else float('nan'):.3f}")

143 plane x prescription x signal evaluations
NOTE: Z values use tight-region B*C/D predictions from the n_eff~1 regime -
indicative ranking only, NOT calibrated significances (see notebook 02).
P4 leakage per signal point (S_region/S_A) and prediction bias vs mu:


  2Mu2E_100GeV_1p2GeV_0p096mm        B/A=- C/A=- D/A=- bias(mu=1)=0.000


  2Mu2E_100GeV_1p2GeV_9p6mm          B/A=0.036 C/A=23.429 D/A=1.357 bias(mu=1)=0.000


  2Mu2E_100GeV_1p2GeV_96p0mm         B/A=0.000 C/A=26.000 D/A=4.000 bias(mu=1)=0.000


  2Mu2E_500GeV_1p2GeV_0p019mm        B/A=0.008 C/A=0.000 D/A=0.000 bias(mu=1)=0.000


...

  2Mu2E_1000GeV_1p2GeV_9p6mm         B/A=0.016 C/A=0.000 D/A=0.000 bias(mu=1)=0.000


  2Mu2E_500GeV_0p25GeV_0p4mm         B/A=0.005 C/A=0.000 D/A=0.000 bias(mu=1)=0.000


  2Mu2E_500GeV_0p25GeV_4p0mm         B/A=0.014 C/A=0.000 D/A=0.000 bias(mu=1)=0.000


  2Mu2E_500GeV_5p0GeV_8p0mm          B/A=0.006 C/A=0.000 D/A=0.000 bias(mu=1)=0.000


  2Mu2E_500GeV_5p0GeV_80p0mm         B/A=0.021 C/A=0.000 D/A=0.000 bias(mu=1)=0.000


In [3]:
# ranking summary: median Z across signal points per plane, worst-case leakage
import collections
agg = collections.defaultdict(list)
for key, r in results.items():
    ch, pname, presc, s = key.split("/", 3)
    lk = [v for v in r["leakage"].values() if np.isfinite(v)]
    agg[f"{ch}/{pname}/{presc}"].append((r["Z"], np.nanmax(lk) if lk else np.nan))
print(f"{'plane':24s} {'median Z':>9s} {'max Z':>7s} {'worst leakage':>14s}")
for k, v in sorted(agg.items(), key=lambda kv: -np.median([x[0] for x in kv[1]])):
    zs = [x[0] for x in v]; ls = [x[1] for x in v]
    print(f"{k:24s} {np.median(zs):9.3f} {max(zs):7.3f} {max(ls):14.3f}")

plane                     median Z   max Z  worst leakage
4mu/Q5_dphi_mjj/i            1.445   3.746            nan
4mu/Q5_dphi_mjj/iii          1.445   3.746            nan
4mu/Q2_iso0_dphi/iii         1.269   3.641            nan
2mu2e/P4_muiso_mjj/i         0.215   1.006            nan
2mu2e/P4_muiso_mjj/iii       0.166   0.977            nan
2mu2e/P1_iso_iso/i           0.046   0.277            nan
2mu2e/P1_iso_iso/iii         0.034   0.266            nan
2mu2e/P8_dphi_mjj/i          0.010   0.064            nan
2mu2e/P8_dphi_mjj/iii        0.010   0.064            nan
2mu2e/P2_muiso_dphi/i        0.002   0.011            nan
2mu2e/P2_muiso_dphi/iii      0.001   0.011            nan


## Per-point detail plots

*(bar charts per plane after execution — kept lean until the numbers exist)*